In [ ]:
# 2026-03-23 run against production
import pandas as pd
from tqdm import tqdm
from normdata.utils import import_from_normdata
from csae_pyutils import gsheet_to_df
from apis_core.apis_metainfo.models import Collection, CollectionType, Uri
from apis_core.apis_entities.models import Person


In [ ]:
df = gsheet_to_df("1pBqzY3uSFUrnYlScCmvroc7t3wAEl_Rt37ilrTw2i30")

In [ ]:
df

In [ ]:
col, _ = Collection.objects.get_or_create(name="S-Fischer")
domain = "sfischer"
col_type, _ = CollectionType.objects.get_or_create(name="Projekt")
col.description = 'To bed onde'
col.collection_type = col_type
col.save()

In [ ]:
print("process persons with gnd")
broken_gnd = []
for i, row in tqdm(df.iterrows()):
    tmp_uri = f"http://sfischer/foo/bar/{i+1}"
    if pd.isna(row["GND-Nummer"]):
        continue
    else:
        gnd = row["GND-Nummer"]
        gnd_uri = f"https://d-nb.info/gnd/{gnd}"
        entity = import_from_normdata(gnd_uri, "person")
        if entity:
            pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_uri, domain=domain)
            pmb_uri.entity = entity
            pmb_uri.save()
            entity.collection.add(col)
        else:
            broken_gnd.append([tmp_uri, gnd_uri])

In [ ]:
print("process persons without gnd")
broken_gnd = []
for i, row in tqdm(df.iterrows()):
    tmp_uri = f"http://sfischer/foo/bar/{i+1}"
    pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_uri, domain=domain)
    entity = pmb_uri.entity
    if pd.isna(row["GND-Nummer"]) and not entity:
        item = {
            "name": row["nachname"]
        }
        if not pd.isna(row["vorname"]):
            item["first_name"] = row["vorname"]
        entity = Person.objects.create(**item)
        pmb_uri.entity = entity
        pmb_uri.save()
        entity.collection.add(col)

In [ ]:
Person.objects.all().count()